# Сранение методов детекции OOD

Датасет: BNCI2014_001

Проверяем детекциюю OOD на Motor Imagery классах и на классах из калибровочных записей
1. Из четырех "left_hand", "right_hand", "feet", "tongue" каждый по очереди становится OOD, остальные три класса как ID
2. Из калибровочных записей нарезаются пробы "rest" (открытые глаза), "eyes_closed", "eyes_move". Они используются по очереди как OOD, 4 класса MI - как ID

Обучение на первой сессии, тест на второй.

К методам из прошлых экспериментов добавлены:  
1. Сочетания ACM+TS, FBCSP, VAR, Bandpower, Microstates с OCSVM, LOF, kNN, IsolationForest, Mahalonobis.
2. FBCSP с голосованием: "FBCSP+Vote" в разных вариантах. 11 моделей обучаются на разных подмножествах (половине) признаков, несогласие используется как оценка OOD.
3. EEGNex+MCdropout - адаптация метода из статьи https://pubmed.ncbi.nlm.nih.gov/42790484/ (там результат median=0.6 для подобного теста использованием каждого mi класса как OOD, но тестирование проводилось на той же сессии)
4. MultiModel: Используются 4 представления: ACM+TS, FBCSP, VAR, Bandpower, каждый со своим классификатором/детектором. Результат получается либо через SoftVote (для оценки OOD используется нормированная энтропия), либо через среднее (после RobustScaler)

## Выводы: 
* Результаты работы методов значительно отличаются в зависимости от выбранного вида OOD, и для разных участников
* Нет какого-то одного метода, который хорошо работает во всех случаях
* Для определения MI и не MI как OOD получились разные лучшие методы
* Модели, которые показывают лучшие средние результаты, могут при этом очень плохо работать на каких-то конкретных участниках. И другие методы на этих участниках могут давать значительно лучший результат.
* Модель с объединением нескольких представлений в данном эксперименте показала лучший средний резлуьтат, при этом, если смотреть на конкретные случаи (участник + вид OOD), то лучший результат часто показывают другие модели

Что может быть интересно поисследовать: 
* Можно ли как-то по тестовым ЭЭГ понять, какие методы лучше сработают для детекции OOD
* Разные способы объединения основных моделей с адаптацией под особенности участника

In [7]:
import os
from pathlib import Path

DATA_ROOT = Path.home() / "mne_data"

os.environ["MNE_DATA"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_BNCI_PATH"] = str(DATA_ROOT)
os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"

EXPERIMENT_DIR = Path("/Users/liza/Projects/vkr/mi-ood")
CACHE_DIR = EXPERIMENT_DIR / "cache"
RESULTS_DIR = EXPERIMENT_DIR / "results"

print(EXPERIMENT_DIR)

/Users/liza/Projects/vkr/mi-ood


In [8]:
from pipelines import ood_pipelines
from processing import *
import numpy as np
from datetime import datetime


In [9]:
experiment_main = "OODs"
experiment_rest = "OODs_rest"

CLASSES = {
    "BNCI2014_001": {
        "mi": ("left_hand", "right_hand", "feet", "tongue"),
        "rest": ("rest", "eyes_closed", "eyes_move"),
    },
}
datasets = list(CLASSES)
COMMON_CHANNELS = None
OOD_METHODS = ['ACM+TS+OCSVM', "ACM+TS+kNN", "ACM+TS+LOF", "ACM+TS+Mahalanobis", "ACM+TS+IsolationForest",
               'FBCSP+IsolationForest', 'FBCSP(nonorm)+IsolationForest', "FBCSP+kNN", "FBCSP+LOF", "FBCSP+OCSVM", "FBCSP+Mahalanobis",
               "FBCSP+Vote", "FBCSP+Vote(entropy)", "FBCSP+Vote(LDA)", "FBCSP+Vote(SVM)", "FBCSP+Vote(ExtraTrees)", "FBCSP+Vote(LR+LDA+SVM)",
               "MultiModel+LOF", "MultiModel+OCSVM", "MultiModel+Mahalanobis",
               "MultiModel+SoftVote", "MultiModel+SoftVote(LDA)", "MultiModel+SoftVote(SVM)", 
               "Log-bandpower+Mahalanobis", "Log-bandpower+kNN", "Log-bandpower+LOF", "Log-bandpower+OCSVM", "Log-bandpower+IsolationForest",
               "Microstates+LOF", "Microstates+kNN", "Microstates+OCSVM", "Microstates+Mahalanobis", "Microstates+IsolationForest",
               "VAR+Mahalanobis", "VAR+kNN", "VAR+LOF", "VAR+OCSVM", "VAR+IsolationForest",
               'ShallowConvNet+kNN', "EEGNet+TempDens",
               "EEGNex+MCdropout" ]



In [10]:
for dataset in datasets:
    print( dataset )
    classes = CLASSES[dataset]["mi"]
    class_rest = CLASSES[dataset]["rest"]
    subjects = get_subjects(dataset) 

    for subject in subjects:
        for ood in classes + class_rest:
            id_classes = tuple(c for c in classes if c != ood)
            experiment = experiment_rest if ood in class_rest else experiment_main
            experiment_type = f"ood_{ood}"
            
            data = None
            for method, pipeline in ood_pipelines.items():
                if method not in OOD_METHODS:
                    continue
                result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": experiment_type, "method": method}
                if result_path(result).exists():
                    continue
                if data is None:
                    data = load_subject_data( dataset, subject, id_classes=id_classes, ood_classes=(ood,), session_index=None, channels=COMMON_CHANNELS)
                train_data, val_data, test_data = data

                model = train(train_data, pipeline)
                result["OOD_AUROC"] = OOD_AUROC(test_data, model, known_classes=id_classes)
                save_result(result)
                print(f"[{datetime.now():%H:%M:%S}] {dataset} s{subject} {ood} {method}: {result['OOD_AUROC']:.3f}")
    
    #display_ood_summary(dataset, experiment_main)
    #display_ood_summary(dataset, experiment_rest )

BNCI2014_001


In [11]:
display_ood_summary(dataset, experiment_main, methods=OOD_METHODS)
display_ood_summary(dataset, experiment_rest, methods=OOD_METHODS)
display_ood_by_subject(dataset, experiment_main, methods=OOD_METHODS)
display_ood_by_subject(dataset, experiment_rest, methods=OOD_METHODS)

BNCI2014_001 OODs: summary


,mean,std,median,iqr
method,,,,
MultiModel+SoftVote(SVM),0.577,0.132,0.607,0.176
MultiModel+SoftVote,0.576,0.106,0.582,0.152
MultiModel+SoftVote(LDA),0.572,0.110,0.592,0.127
EEGNet+TempDens,0.556,0.198,0.534,0.233
MultiModel+LOF,0.549,0.096,0.526,0.093
FBCSP+LOF,0.549,0.151,0.540,0.209
ACM+TS+Mahalanobis,0.547,0.121,0.523,0.121
ACM+TS+kNN,0.546,0.130,0.536,0.116
ACM+TS+LOF,0.546,0.115,0.530,0.110


BNCI2014_001 OODs_rest: summary


,mean,std,median,iqr
method,,,,
MultiModel+Mahalanobis,0.754,0.203,0.785,0.322
MultiModel+LOF,0.750,0.224,0.768,0.248
VAR+Mahalanobis,0.742,0.191,0.814,0.245
MultiModel+OCSVM,0.734,0.220,0.744,0.283
ACM+TS+LOF,0.717,0.222,0.753,0.359
ACM+TS+Mahalanobis,0.708,0.217,0.751,0.330
ACM+TS+OCSVM,0.707,0.222,0.748,0.348
ACM+TS+kNN,0.702,0.218,0.725,0.349
VAR+LOF,0.679,0.198,0.664,0.225


# BNCI2014_001 OODs: OOD_AUROC


# BNCI2014_001 OODs_rest: OOD_AUROC


In [12]:
display_ood_groups(dataset, experiment_main, group="representation")
display_ood_groups(dataset, experiment_main, group="detector")

display_ood_groups(dataset, experiment_rest, group="representation")
display_ood_groups(dataset, experiment_rest, group="detector")

BNCI2014_001 OODs: representation, Summary


ood_class,mean,feet,left_hand,right_hand,tongue
representation,,,,,
ACM+TS,0.542,0.537,0.464,0.523,0.642
Microstates,0.530,0.541,0.512,0.509,0.558
FBCSP,0.529,0.574,0.437,0.541,0.564
Log-bandpower,0.524,0.518,0.535,0.483,0.562
VAR,0.505,0.497,0.497,0.510,0.519


BNCI2014_001 OODs: representation, By subject


BNCI2014_001 OODs: detector, Summary


ood_class,mean,feet,left_hand,right_hand,tongue
detector,,,,,
LOF,0.531,0.525,0.511,0.535,0.553
Mahalanobis,0.530,0.551,0.472,0.523,0.575
kNN,0.526,0.547,0.477,0.507,0.572
OCSVM,0.525,0.512,0.505,0.505,0.579
IsolationForest,0.518,0.532,0.480,0.496,0.565


BNCI2014_001 OODs: detector, By subject


... (2 outputs left out)

BNCI2014_001 OODs_rest: representation, By subject


BNCI2014_001 OODs_rest: detector, Summary


ood_class,mean,eyes_closed,eyes_move,rest
detector,,,,
OCSVM,0.663,0.581,0.618,0.789
LOF,0.642,0.579,0.606,0.740
kNN,0.642,0.578,0.596,0.751
IsolationForest,0.630,0.561,0.578,0.749
Mahalanobis,0.623,0.582,0.589,0.698


BNCI2014_001 OODs_rest: detector, By subject
